# 9.16 模型給機率後，程式怎樣決策？


假設客服只能做兩件事：直接開啟卡片進度頁，或先問一個澄清問題。模型判斷「該開卡片進度頁」的機率是0.8；這是這裡另設的數字，不是9.11示意的0.844，選0.8是因為它剛好落在下面算出的門檻上。要不要直接開，不能只從0.8決定；開錯頁與多問一次的成本也要先約定。

用人工訂的成本說明：開錯頁的成本是5單位，開對是0；先澄清固定花1單位，並暫時假設問一次就能解決這一輪的不確定。若 p 是經過校準、可以用來估算答對機會的機率，直接開頁的**期望成本**是 `5*(1-p)`：開錯的成本5乘上開錯的機率 1−p，再加上開對的成本0乘上 p；澄清則固定是1。在 p 大於0.8時，前者較小；剛好相等時由事先訂的規則處理。這個門檻只屬於這張簡化的成本表；換成轉帳這類做錯成本大得多的動作，成本表不同，門檻也必須重算。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
for p in [0.6, 0.8, 0.9]:
    answer_cost = 5 * (1 - p)
    # 在這個示例的浮點容差內視作平手，依預定規則選澄清。
    action = "直接開頁" if answer_cost < 1 - 1e-9 else "澄清"
    print(p, round(answer_cost, 3), action)

程式在0.6、0.8選澄清，在0.9直接開頁。0.8剛好在門檻上，但電腦算出的`5 * (1 - 0.8)`是0.9999999999999998，比1少一點點；若直接比`< 1`，平手會被誤判成直接開頁。所以比較時減掉很小的容差`1e-9`，把它當成平手，照事先規則選澄清；`round`只讓印出的成本好讀。

實際服務若澄清也可能失敗，就要把後續成功機會及成本納入；若模型沒有校準，或把[9.15](https://birdhackor.github.io/tiny-perceptron-vlm/9.15.html)的集中度當成 p，就不能保證期望成本算得正確。部署前應在[9.14](https://birdhackor.github.io/tiny-perceptron-vlm/9.14.html)的決策資料上實測這條規則，最後在測試資料上同時報三個數：自動處理的比例（覆蓋率）、自動處理那些題的錯誤率，以及總成本；不能只報「留下來自動處理的題很準」。

這種安排把工作分成兩層：模型從材料預測，程式依任務成本與授權決定動作。可以在適當時查資料、呼叫工具或轉人工，但每種動作有自己的條件；本節只做有限模擬，不執行真實個人資料操作。第19章的 Auto 也使用這個分工（[19.17](https://birdhackor.github.io/tiny-perceptron-vlm/19.17.html)，讀到第19章再看）：先由機率判斷器判斷題目是否支援、資訊是否足夠，再依事先訂好的規則選解題預算。

另一個可對照的實務案例是 **DeskMind**：一個開源的電腦操作助理，作者仿照 Jev 的介面訓練了自己的小型決策模型，並公開訓練說明。說明裡的做法包括：拿現成的語言模型用 LoRA（[8.8](https://birdhackor.github.io/tiny-perceptron-vlm/8.8.html)）調整、讓教師模型提供每題的機率分布，以及每題隨機打亂候選順序（對照[9.12](https://birdhackor.github.io/tiny-perceptron-vlm/9.12.html)），適合對照本章的資料約定。它是獨立開發者的專案，不是 TypeSafe 的官方說明，也沒有提供足以重現 Jev 專有訓練的全部細節。小型自測全對，也不能替代按家族切出的校準與泛化評估。本課第19章使用自製有限世界及隨機初始化的參數，讓標籤與限制都可以查清楚。

練習：把程式裡的`1 - 1e-9`改成`0.5 - 1e-9`，也就是澄清成本降成0.5。先預測門檻：解 `5*(1-p)<0.5` 得 p>0.9，所以三個 p 都選澄清，其中0.9剛好平手，照規則也選澄清；再執行核對。多問一次越便宜，程式就有理由更常澄清；這是成本選擇，不代表模型忽然變得不確定。

案例：[DeskMind launch](https://deskmind.dev/zh/blog/launch/)、[公開 training 說明](https://github.com/deskmind-ai/brain/blob/main/docs/training.md)（讀取2026-10-18）。
